In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/Audible_Catlog.csv")
df_advance = pd.read_csv("../data/raw/Audible_Catlog_Advanced_Features.csv")

print("Dataset 1:", df.shape)
print("Dataset 2:", df_advance.shape)

Dataset 1: (6368, 5)
Dataset 2: (4464, 8)


Remove Duplicates

In [2]:
print("Dataset 1 exact duplicates:", df.duplicated().sum())
print("Dataset 2 exact duplicates:", df_advance.duplicated().sum())

Dataset 1 exact duplicates: 929
Dataset 2 exact duplicates: 168


In [3]:
df_clean = df.drop_duplicates().copy()
df_advance_clean = df_advance.drop_duplicates().copy()

print("Dataset 1 after cleaning:", df_clean.shape)
print("Dataset 2 after cleaning:", df_advance_clean.shape)

Dataset 1 after cleaning: (5439, 5)
Dataset 2 after cleaning: (4296, 8)


Missing Values analysis

In [4]:
print("Dataset 1 Missing Values:")
display(df_clean.isnull().sum())

print("\nDataset 2 Missing Values:")
display(df_advance_clean.isnull().sum())

Dataset 1 Missing Values:


Book Name              0
Author                 0
Rating                 0
Number of Reviews    524
Price                  2
dtype: int64


Dataset 2 Missing Values:


Book Name              0
Author                 0
Rating                 0
Number of Reviews    405
Price                  0
Description            6
Listening Time         0
Ranks and Genre        0
dtype: int64

In [5]:
print("Dataset 1 Rating = -1:", (df_clean["Rating"] == -1).sum())
print("Dataset 2 Rating = -1:", (df_advance_clean["Rating"] == -1).sum())

Dataset 1 Rating = -1: 524
Dataset 2 Rating = -1: 405


In [6]:
print("Dataset 1 - Rating -1 but Reviews present:",
      df_clean[(df_clean["Rating"] == -1) &
               (df_clean["Number of Reviews"].notna())].shape[0])

print("Dataset 2 - Rating -1 but Reviews present:",
      df_advance_clean[(df_advance_clean["Rating"] == -1) &
                        (df_advance_clean["Number of Reviews"].notna())].shape[0])

Dataset 1 - Rating -1 but Reviews present: 0
Dataset 2 - Rating -1 but Reviews present: 0


In [7]:
df_clean["Rating"] = df_clean["Rating"].replace(-1, pd.NA)
df_advance_clean["Rating"] = df_advance_clean["Rating"].replace(-1, pd.NA)

print("Dataset 1 missing ratings:", df_clean["Rating"].isna().sum())
print("Dataset 1 missing ratings:", df_advance_clean["Rating"].isna().sum())

Dataset 1 missing ratings: 524
Dataset 1 missing ratings: 405


In [8]:
print("Listening Time = -1:",
      (df_advance_clean["Listening Time"] == "-1").sum())

print("Ranks and Genre = -1:",
      (df_advance_clean["Ranks and Genre"] == "-1").sum())

Listening Time = -1: 2168
Ranks and Genre = -1: 2168


In [9]:
df_advance_clean["Listening Time"] = df_advance_clean["Listening Time"].replace("-1", pd.NA)
df_advance_clean["Ranks and Genre"] = df_advance_clean["Ranks and Genre"].replace("-1", pd.NA)

print("Missing Listening Time:",
      df_advance_clean["Listening Time"].isna().sum())

print("Missing Ranks and Genre:",
      df_advance_clean["Ranks and Genre"].isna().sum())

Missing Listening Time: 2168
Missing Ranks and Genre: 2168


In [10]:
error_patterns = (
    r"Sorry, we just need to make sure|"
    r"Oops! It's rush hour|"
    r"By completing your purchase"
)

error_descriptions = df_advance_clean[
    df_advance_clean["Description"].str.contains(
        error_patterns, case=False, na=False, regex=True
    )
]

print("Descriptions containing scraping/purchase errors:",
      len(error_descriptions))

Descriptions containing scraping/purchase errors: 1113


In [11]:
df_advance_clean.loc[
    df_advance_clean["Description"].str.contains(
        error_patterns, case=False, na=False, regex=True
    ),
    "Description"
] = pd.NA

print("Remaining error descriptions:",
      df_advance_clean["Description"].str.contains(
          error_patterns, case=False, na=False, regex=True
      ).sum())

print("Total missing descriptions:",
      df_advance_clean["Description"].isna().sum())

Remaining error descriptions: 0
Total missing descriptions: 1119


In [12]:
text_columns_d1 = ["Book Name", "Author"]
text_columns_d2 = ["Book Name", "Author", "Description"]

for col in text_columns_d1:
    df_clean[col] = df_clean[col].str.strip()

for col in text_columns_d2:
    df_advance_clean[col] = df_advance_clean[col].str.strip()

print("Dataset 1 whitespace normalization complete.")
print("Dataset 2 whitespace normalization complete.")

Dataset 1 whitespace normalization complete.
Dataset 2 whitespace normalization complete.


In [13]:
print("Dataset 1 empty strings:")
display((df_clean[["Book Name", "Author"]] == "").sum())

print("\nDataset 2 empty strings:")
display((df_advance_clean[["Book Name", "Author", "Description"]] == "").sum())

Dataset 1 empty strings:


Book Name    0
Author       0
dtype: int64


Dataset 2 empty strings:


Book Name      0
Author         0
Description    2
dtype: int64

In [14]:
df_advance_clean["Description"] = df_advance_clean["Description"].replace("", pd.NA)

print("Empty Description strings:",
      (df_advance_clean["Description"] == "").sum())

print("Missing Descriptions:",
      df_advance_clean["Description"].isna().sum())

Empty Description strings: 0
Missing Descriptions: 1121


In [15]:
print("Dataset 1 missing Price:", df_clean["Price"].isna().sum())
print("Dataset 1 zero Price:", (df_clean["Price"] == 0).sum())

print("\nDataset 2 missing Price:", df_advance_clean["Price"].isna().sum())
print("Dataset 2 zero Price:", (df_advance_clean["Price"] == 0).sum())

Dataset 1 missing Price: 2
Dataset 1 zero Price: 96

Dataset 2 missing Price: 0
Dataset 2 zero Price: 74


In [16]:
display(
    df_advance_clean["Listening Time"]
    .dropna()
    .head(20)
)

0     10 hours and 54 minutes
1      3 hours and 23 minutes
2      5 hours and 17 minutes
3      5 hours and 35 minutes
4      6 hours and 25 minutes
5     15 hours and 18 minutes
6        1 hour and 6 minutes
7     17 hours and 48 minutes
8       6 hours and 9 minutes
9      11 hours and 4 minutes
10     10 hours and 6 minutes
11     7 hours and 19 minutes
12    10 hours and 54 minutes
13      8 hours and 5 minutes
14                    4 hours
15      20 hours and 1 minute
16                   14 hours
17    71 hours and 57 minutes
18     19 hours and 3 minutes
19    10 hours and 31 minutes
Name: Listening Time, dtype: object

In [17]:
import re

def convert_to_minutes(value):
    if pd.isna(value):
        return pd.NA

    hours = re.search(r"(\d+)\s+hours?", value)
    minutes = re.search(r"(\d+)\s+minutes?", value)

    total_minutes = 0

    if hours:
        total_minutes += int(hours.group(1)) * 60

    if minutes:
        total_minutes += int(minutes.group(1))

    return total_minutes


df_advance_clean["Listening_Time_Minutes"] = (
    df_advance_clean["Listening Time"].apply(convert_to_minutes)
)

display(
    df_advance_clean[
        ["Listening Time", "Listening_Time_Minutes"]
    ].head(20)
)

,Listening Time,Listening_Time_Minutes
0,10 hours and 54 minutes,654
1,3 hours and 23 minutes,203
2,5 hours and 17 minutes,317
3,5 hours and 35 minutes,335
4,6 hours and 25 minutes,385
5,15 hours and 18 minutes,918
6,1 hour and 6 minutes,66
7,17 hours and 48 minutes,1068
8,6 hours and 9 minutes,369
9,11 hours and 4 minutes,664


In [18]:
print("Missing Listening Time (minutes):",
      df_advance_clean["Listening_Time_Minutes"].isna().sum())

print("Minimum duration:",
      df_advance_clean["Listening_Time_Minutes"].min())

print("Maximum duration:",
      df_advance_clean["Listening_Time_Minutes"].max())

display(
    df_advance_clean[
        ["Listening Time", "Listening_Time_Minutes"]
    ].sample(10, random_state=42)
)

Missing Listening Time (minutes): 2168
Minimum duration: 0
Maximum duration: 7591


,Listening Time,Listening_Time_Minutes
3325,<NA>,<NA>
3847,<NA>,<NA>
4095,<NA>,<NA>
305,6 hours and 58 minutes,418
1672,<NA>,<NA>
4425,<NA>,<NA>
1126,7 hours and 32 minutes,452
697,4 hours and 37 minutes,277
2663,<NA>,<NA>
1670,<NA>,<NA>


In [19]:
print("Dataset 1 data types:")
display(df_clean.dtypes)

print("\nDataset 2 data types:")
display(df_advance_clean.dtypes)

Dataset 1 data types:


Book Name             object
Author                object
Rating                object
Number of Reviews    float64
Price                float64
dtype: object


Dataset 2 data types:


Book Name                  object
Author                     object
Rating                     object
Number of Reviews         float64
Price                       int64
Description                object
Listening Time             object
Ranks and Genre            object
Listening_Time_Minutes     object
dtype: object

In [20]:
# Convert numeric columns to appropriate data types

numeric_columns_d1 = ["Rating", "Number of Reviews", "Price"]
numeric_columns_d2 = ["Rating", "Number of Reviews", "Price", "Listening_Time_Minutes"]

for col in numeric_columns_d1:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

for col in numeric_columns_d2:
    df_advance_clean[col] = pd.to_numeric(df_advance_clean[col], errors="coerce")

print("Dataset 1 data types:")
display(df_clean.dtypes)

print("\nDataset 2 data types:")
display(df_advance_clean.dtypes)

Dataset 1 data types:


Book Name             object
Author                object
Rating               float64
Number of Reviews    float64
Price                float64
dtype: object


Dataset 2 data types:


Book Name                  object
Author                     object
Rating                    float64
Number of Reviews         float64
Price                       int64
Description                object
Listening Time             object
Ranks and Genre            object
Listening_Time_Minutes    float64
dtype: object

In [21]:
print("Dataset 1 Missing Values:")
display(df_clean.isnull().sum())

print("\nDataset 2 Missing Values:")
display(df_advance_clean.isnull().sum())

Dataset 1 Missing Values:


Book Name              0
Author                 0
Rating               524
Number of Reviews    524
Price                  2
dtype: int64


Dataset 2 Missing Values:


Book Name                    0
Author                       0
Rating                     405
Number of Reviews          405
Price                        0
Description               1121
Listening Time            2168
Ranks and Genre           2168
Listening_Time_Minutes    2168
dtype: int64

In [22]:
print("Dataset 1 remaining duplicates:", df_clean.duplicated().sum())

print("Dataset 2 remaining duplicates:", df_advance_clean.duplicated().sum())

Dataset 1 remaining duplicates: 0
Dataset 2 remaining duplicates: 3


In [23]:
duplicate_rows = df_advance_clean[
    df_advance_clean.duplicated(keep=False)
]

display(duplicate_rows)

,Book Name,Author,Rating,Number of Reviews,Price,Description,Listening Time,Ranks and Genre,Listening_Time_Minutes
3937,The Executive Voice Trainer,Jeffrey Jacobi,NaN,NaN,702,<NA>,<NA>,<NA>,NaN
3973,The Executive Voice Trainer,Jeffrey Jacobi,NaN,NaN,702,<NA>,<NA>,<NA>,NaN
4025,English in Situations - New Edition - Business...,Dorota Guzik,NaN,NaN,334,<NA>,<NA>,<NA>,NaN
4039,English in Situations - New Edition - Business...,Dorota Guzik,NaN,NaN,334,<NA>,<NA>,<NA>,NaN
4203,Children around the Prophet: How Muhammad (Pea...,Dr. Hesham Al-Awadi,NaN,NaN,937,<NA>,<NA>,<NA>,NaN
4214,Children around the Prophet: How Muhammad (Pea...,Dr. Hesham Al-Awadi,NaN,NaN,937,<NA>,<NA>,<NA>,NaN


In [24]:
df_advance_clean = df_advance_clean.drop_duplicates().copy()

print("Dataset 2 shape after removing remaining duplicates:", df_advance_clean.shape)
print("Remaining duplicates:", df_advance_clean.duplicated().sum())

Dataset 2 shape after removing remaining duplicates: (4293, 9)
Remaining duplicates: 0


In [25]:
print("Final Dataset 1 Shape:", df_clean.shape)
print("Final Dataset 2 Shape:", df_advance_clean.shape)

Final Dataset 1 Shape: (5439, 5)
Final Dataset 2 Shape: (4293, 9)


In [26]:
df_clean.to_csv("../data/processed/audible_clean.csv", index=False)

df_advance_clean.to_csv(
    "../data/processed/audible_advanced_clean.csv",
    index=False
)

print("Cleaned datasets saved successfully!")

Cleaned datasets saved successfully!
